# SportsGraph — Phase 2: YOLO Ball Specialist Fine-Tuning

Self-contained Kaggle runner for training the **1-class soccer ball detector** (`yolo_ball_best.pt`) on GPU.

### Dual-Detector Architecture
- **Model 1 (Role Specialist)**: `best.pt` (Player: 0.985 AP, Goalkeeper: 0.980 AP, Referee: 0.918 AP)
- **Model 2 (Ball Specialist)**: Trained here on **4,250+ combined frames** specifically for small-object ball detection
- **Combined Dataset**: Ingests `football-ball-detection` (3,956 train images) + `football-players-detection` (297 ball-filtered train images)
- **Evaluation Benchmark Target**: Exceed our baseline on the 500-frame test set (**0.4316 AP**)

### 1. Dependencies & Environment Setup

In [ ]:
!pip install --upgrade ultralytics pyyaml pillow pandas

import shutil
import zipfile
from pathlib import Path

import yaml

WORKING_DIR = Path("/kaggle/working")
DATA_DIR = WORKING_DIR / "data" / "ball_combined"
MODELS_DIR = WORKING_DIR / "models" / "ball_specialist"

print("Environment initialized.")

### 2. Auto-Discover & Ingest Datasets (Handles Zips or Extracted Folders)
Automatically discovers `ball-data` in `/kaggle/input`, whether Kaggle left them as `.zip` files or auto-extracted them into directories.

In [ ]:
input_path = Path("/kaggle/input")
IMG_EXTS = (".jpg", ".png", ".jpeg")

print("--- Inspecting /kaggle/input ---")
if input_path.exists():
    for p in sorted(input_path.iterdir()):
        print(f"Dataset: {p.name} ({'dir' if p.is_dir() else 'file'})")
        if p.is_dir():
            sub_items = list(p.iterdir())
            for s in sub_items[:8]:
                print(f"  └── {s.name}")
            if len(sub_items) > 8:
                print(f"  └── ... ({len(sub_items) - 8} more items)")
else:
    print("Warning: /kaggle/input does not exist!")


def filter_ball_label_content(content: str, ball_class_id: int = 0) -> str:
    filtered_lines = []
    for line in content.strip().splitlines():
        parts = line.strip().split()
        if parts and parts[0] == str(ball_class_id):
            filtered_lines.append("0 " + " ".join(parts[1:]))
    return "\n".join(filtered_lines) + ("\n" if filtered_lines else "")


def unpack_ball_source(source: Path, target_dir: Path) -> dict[str, int]:
    counts = {"train": 0, "valid": 0}
    target_dir.mkdir(parents=True, exist_ok=True)

    if source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source, "r") as z:
            for member in z.namelist():
                for s in ["train", "valid"]:
                    is_img = member.startswith(f"{s}/images/") and member.lower().endswith(IMG_EXTS)
                    if is_img:
                        s_img = target_dir / s / "images"
                        s_lbl = target_dir / s / "labels"
                        s_img.mkdir(parents=True, exist_ok=True)
                        s_lbl.mkdir(parents=True, exist_ok=True)

                        fname = Path(member).name
                        (s_img / fname).write_bytes(z.read(member))
                        counts[s] += 1

                        lbl_m = f"{s}/labels/{Path(member).stem}.txt"
                        dest_lbl = s_lbl / f"{Path(member).stem}.txt"
                        if lbl_m in z.namelist():
                            dest_lbl.write_bytes(z.read(lbl_m))
                        else:
                            dest_lbl.write_text("")
    elif source.is_dir():
        for s in ["train", "valid"]:
            cand_img_dirs = list(source.rglob(f"{s}/images")) + list(source.rglob(f"{s}"))
            cand_img_dirs = [d for d in cand_img_dirs if d.is_dir() and any(d.glob("*.jpg"))]
            if cand_img_dirs:
                s_img = target_dir / s / "images"
                s_lbl = target_dir / s / "labels"
                s_img.mkdir(parents=True, exist_ok=True)
                s_lbl.mkdir(parents=True, exist_ok=True)
                src_dir = cand_img_dirs[0]
                lbl_parent = src_dir.parent / "labels"
                src_lbl = lbl_parent if lbl_parent.exists() else src_dir
                for img in src_dir.glob("*.*"):
                    if img.suffix.lower() in IMG_EXTS:
                        shutil.copy2(img, s_img / img.name)
                        counts[s] += 1
                        lbl = src_lbl / f"{img.stem}.txt"
                        if lbl.exists():
                            shutil.copy2(lbl, s_lbl / f"{img.stem}.txt")
                        else:
                            (s_lbl / f"{img.stem}.txt").write_text("")
    return counts


def merge_players_source(source: Path, target_dir: Path) -> int:
    dest_img = target_dir / "train" / "images"
    dest_lbl = target_dir / "train" / "labels"
    dest_img.mkdir(parents=True, exist_ok=True)
    dest_lbl.mkdir(parents=True, exist_ok=True)
    count = 0

    if source.is_file() and source.suffix.lower() == ".zip":
        with zipfile.ZipFile(source, "r") as z:
            for member in z.namelist():
                is_img = member.startswith("train/images/") and member.lower().endswith(IMG_EXTS)
                if is_img:
                    fname = f"players_{Path(member).name}"
                    (dest_img / fname).write_bytes(z.read(member))
                    lbl_m = f"train/labels/{Path(member).stem}.txt"
                    dest_l = dest_lbl / f"players_{Path(member).stem}.txt"
                    if lbl_m in z.namelist():
                        raw = z.read(lbl_m).decode("utf-8")
                        dest_l.write_text(filter_ball_label_content(raw, ball_class_id=0))
                    else:
                        dest_l.write_text("")
                    count += 1
    elif source.is_dir():
        cand_dirs = list(source.rglob("train/images")) + list(source.rglob("images"))
        cand_dirs = [d for d in cand_dirs if d.is_dir() and any(d.glob("*.jpg"))]
        if cand_dirs:
            src_img = cand_dirs[0]
            lbl_parent = src_img.parent / "labels"
            src_lbl = lbl_parent if lbl_parent.exists() else src_img
            for img in src_img.glob("*.*"):
                if img.suffix.lower() in IMG_EXTS:
                    fname = f"players_{img.name}"
                    shutil.copy2(img, dest_img / fname)
                    lbl = src_lbl / f"{img.stem}.txt"
                    dest_l = dest_lbl / f"players_{img.stem}.txt"
                    if lbl.exists():
                        raw = lbl.read_text(encoding="utf-8")
                        dest_l.write_text(filter_ball_label_content(raw, ball_class_id=0))
                    else:
                        dest_l.write_text("")
                    count += 1
    return count


# Locate candidate ball and player sources
all_candidates = list(input_path.rglob("*.zip")) + [d for d in input_path.rglob("*") if d.is_dir()]
ball_sources = [
    c for c in all_candidates if "ball" in c.name.lower() and (c.is_file() or any(c.rglob("*.jpg")))
]
player_sources = [
    c
    for c in all_candidates
    if "player" in c.name.lower() and (c.is_file() or any(c.rglob("*.jpg")))
]

if not ball_sources:
    ball_sources = [p for p in input_path.iterdir() if p.is_dir() or p.suffix.lower() == ".zip"]
    if not ball_sources:
        raise FileNotFoundError("No dataset found in /kaggle/input!")

print(f"Using primary ball source: {ball_sources[0]}")
counts = unpack_ball_source(ball_sources[0], DATA_DIR)
print(f"Unpacked primary ball dataset: {counts['train']} train, {counts['valid']} val")

if player_sources:
    print(f"Merging player dataset ball annotations from: {player_sources[0]}")
    added = merge_players_source(player_sources[0], DATA_DIR)
    print(f"Added {added} ball-filtered frames from players dataset.")
    total_train = counts["train"] + added
else:
    total_train = counts["train"]
    print("Note: player dataset source not found; proceeding with primary ball dataset.")

print(f"Final combined training corpus: {total_train} images in {DATA_DIR}!")

### 3. Generate Manifest (Single-Class: Ball)

In [ ]:
train_images = sorted((DATA_DIR / "train" / "images").glob("*.jpg")) + sorted(
    (DATA_DIR / "train" / "images").glob("*.png")
)
val_images = sorted((DATA_DIR / "valid" / "images").glob("*.jpg")) + sorted(
    (DATA_DIR / "valid" / "images").glob("*.png")
)

train_txt = DATA_DIR / "train.txt"
val_txt = DATA_DIR / "val.txt"

train_txt.write_text("\n".join(str(p.resolve().as_posix()) for p in train_images) + "\n")
val_txt.write_text("\n".join(str(p.resolve().as_posix()) for p in val_images) + "\n")

manifest = {
    "path": str(DATA_DIR.as_posix()),
    "train": str(train_txt.resolve().as_posix()),
    "val": str(val_txt.resolve().as_posix()),
    "nc": 1,
    "names": ["ball"],
}

manifest_file = DATA_DIR / "dataset.yaml"
with open(manifest_file, "w") as f:
    yaml.safe_dump(manifest, f, sort_keys=False)

print(f"Manifest ready: {len(train_images)} train, {len(val_images)} val -> {manifest_file}")

### 4. Train Ball Specialist on GPU

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

train_args = {
    "data": str(manifest_file.resolve().as_posix()),
    "epochs": 50,
    "imgsz": 1280,
    "batch": 8,
    "device": "0",
    "project": str(MODELS_DIR.parent.as_posix()),
    "name": MODELS_DIR.name,
    "exist_ok": True,
    "mosaic": 1.0,
    "mixup": 0.15,
    "hsv_h": 0.015,
    "hsv_s": 0.7,
    "hsv_v": 0.4,
    "fliplr": 0.5,
    "verbose": True,
}

print("Starting Ball Specialist fine-tuning on GPU...")
results = model.train(**train_args)

### 5. Verify & Download Trained Weights

In [ ]:
best_weights = MODELS_DIR / "weights" / "best.pt"
if best_weights.exists():
    size_mb = best_weights.stat().st_size / (1024 * 1024)
    print("Training completed successfully!")
    print(f"Best ball weights: {best_weights} ({size_mb:.2f} MB)")
    print("\n--- Next Steps ---")
    print("1. Download best.pt from the Kaggle Output browser on the right.")
    print("2. Save it locally as: models/ball_specialist/weights/best.pt")
    cmd = (
        "uv run python -m sportsgraph.detection.run_baseline_eval "
        "--images-dir data/raw/ball_test/images "
        "--labels-dir data/raw/ball_test/labels "
        "--classes-file data/raw/ball_test/classes.txt "
        "--model models/ball_specialist/weights/best.pt"
    )
    print(f"3. Run local evaluation to verify against the 0.4316 baseline:\n   {cmd}")
else:
    print("Weights file not found at expected location. Check training logs above.")